# HMDA 2024 Responsible ML Capstone

## 1. Responsible ML Framing

This notebook studies a binary prediction task using HMDA 2024 loan application data: approve (`action_taken` in `1`, `2`) versus deny (`action_taken` equal to `3`). The model is intended to support loan approval decisions, so the system context is not purely technical; these decisions affect access to credit and can create meaningful financial and social consequences.

**Stakeholders:** loan applicants, financial institutions, and regulators.

**Conflicting interests:** lenders want efficient and accurate decisions with controlled risk, applicants want fair access to credit and protection from unjust denial, and regulators want compliance, transparency, and non-discrimination.

**Risk vs benefit:** a useful model can improve consistency and provide a reproducible benchmark for approval decisions, but a poor model can deny credit unfairly, amplify disparities across groups, and create regulatory or reputational harm.

**Failure definition:** failure means more than low predictive quality. It also includes unfair subgroup behavior, excessive harmful errors, or a decision process that benefits institutions while disproportionately harming applicants.

**What are we optimizing?** This notebook does not optimize for accuracy alone. The objective is to balance predictive performance, with `AUC` as a core metric, against fairness across demographic groups.

**Who benefits vs who is harmed?** A better model may help institutions make more consistent decisions and may benefit applicants if errors and disparities are reduced. A worse model can especially harm applicants who are wrongly denied and can expose institutions to fairness and compliance risk.

## Data Requirement Note

Place `2024_lar.zip` in the project root before running this notebook. The notebook assumes local file access and does not download the raw HMDA archive automatically.

## 2. Environment and Imports

This section imports the libraries used in the notebook and defines the paths for the raw HMDA archive, extracted text file, and filtered parquet dataset.

In [13]:
from pathlib import Path
from zipfile import ZipFile

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define the project paths used throughout the notebook.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ZIP_PATH = PROJECT_ROOT / "2024_lar.zip"
TXT_PATH = PROJECT_ROOT / "2024_lar.txt"
PARQUET_PATH = DATA_DIR / "hmda_filtered.parquet"
RAW_MEMBER_NAME = "2024_lar.txt"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Extract the raw text file only if it is not already available locally.
if not TXT_PATH.exists():
    if not ZIP_PATH.exists():
        raise FileNotFoundError("Expected either 2024_lar.txt or 2024_lar.zip in the project root.")
    print("Extracting 2024_lar.txt from the zip archive. This can take a while on the first run...")
    with ZipFile(ZIP_PATH) as zf:
        zf.extract(RAW_MEMBER_NAME, path=PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw HMDA text file: {TXT_PATH}")
print(f"Filtered parquet output: {PARQUET_PATH}")

Project root: /Users/hsuan/Documents/responsible_ML/Capstone
Raw HMDA text file: /Users/hsuan/Documents/responsible_ML/Capstone/2024_lar.txt
Filtered parquet output: /Users/hsuan/Documents/responsible_ML/Capstone/data/hmda_filtered.parquet


## 3. Data Source and Problem Framing

This notebook uses the 2024 HMDA Loan/Application Register data. The prediction task is binary classification: approve (`action_taken` in `1`, `2`) versus deny (`action_taken` equal to `3`). The model is meant to support approval decisions, not to predict default risk or later loan performance.

Reference pages:
- [HMDA 2024 dynamic national loan-level dataset](https://ffiec.cfpb.gov/data-publication/dynamic-national-loan-level-dataset/2024)
- [HMDA loan/application register data fields documentation](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)

## 4. Raw Data Inspection with DuckDB

DuckDB is used to inspect the raw pipe-delimited HMDA file before filtering so the notebook works from the actual schema and field names.

In [14]:
def sql_path(path: Path) -> str:
    return path.as_posix().replace("'", "''")

con = duckdb.connect()

# Preview the raw schema directly from the HMDA text file.
schema_preview = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_csv_auto(
        '{sql_path(TXT_PATH)}',
        delim='|',
        header=True,
        sample_size=20000
    )
    """
).df()

display(schema_preview.head(25))
print(f"Total number of columns in the raw HMDA file: {len(schema_preview)}")

,column_name,column_type,null,key,default,extra
0,activity_year,BIGINT,YES,None,None,None
1,lei,VARCHAR,YES,None,None,None
2,derived_msa_md,BIGINT,YES,None,None,None
3,state_code,VARCHAR,YES,None,None,None
4,county_code,VARCHAR,YES,None,None,None
5,census_tract,VARCHAR,YES,None,None,None
6,conforming_loan_limit,VARCHAR,YES,None,None,None
7,derived_loan_product_type,VARCHAR,YES,None,None,None
8,derived_dwelling_category,VARCHAR,YES,None,None,None
9,derived_ethnicity,VARCHAR,YES,None,None,None


Total number of columns in the raw HMDA file: 99


## 5. Filter and Export a Smaller Dataset

This step keeps only the rows and columns needed for the modeling workflow and saves the result as `data/hmda_filtered.parquet` for reuse.

In [15]:
selected_columns = [
    "action_taken",
    "property_value",
    "income",
    "tract_population",
    "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income",
    "tract_to_msa_income_percentage",
    "state_code",
    "derived_loan_product_type",
    "derived_dwelling_category",
    "loan_purpose",
    "lien_status",
    "occupancy_type",
    "applicant_age",
    "derived_race",
    "derived_sex",
    "derived_ethnicity",
]

selected_columns_sql = ",\n        ".join(selected_columns)

# Create a smaller parquet dataset that keeps only the baseline modeling fields.
con.execute(
    f"""
    COPY (
        SELECT
            {selected_columns_sql}
        FROM read_csv_auto(
            '{sql_path(TXT_PATH)}',
            delim='|',
            header=True,
            all_varchar=True,
            sample_size=20000
        )
        WHERE action_taken IN ('1', '2', '3')
          AND derived_sex IN ('Male', 'Female')
    ) TO '{sql_path(PARQUET_PATH)}' (FORMAT PARQUET)
    """
)

filtered_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{sql_path(PARQUET_PATH)}')"
).df()
filtered_row_count = con.execute(
    f"SELECT COUNT(*) AS row_count FROM read_parquet('{sql_path(PARQUET_PATH)}')"
).fetchone()[0]
filtered_column_count = len(filtered_schema)

print(f"Filtered parquet written to: {PARQUET_PATH}")
print(f"Rows kept for modeling: {filtered_row_count:,}")
print(f"Total number of columns in the filtered dataset: {filtered_column_count}")
print(f"Total number of rows in the filtered dataset: {filtered_row_count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtered parquet written to: /Users/hsuan/Documents/responsible_ML/Capstone/data/hmda_filtered.parquet
Rows kept for modeling: 4,886,502
Total number of columns in the filtered dataset: 17
Total number of rows in the filtered dataset: 4,886,502


## 6. Load the Filtered Data into pandas

The parquet file becomes the main analysis dataset for the rest of the notebook. This section loads it into pandas, previews a few rows, shows the dataset size, and exports a small CSV preview.

In [16]:
df = pd.read_parquet(PARQUET_PATH)
preview_csv_path = DATA_DIR / "hmda_filtered_preview.csv"
preview_rows = 20

print("Parquet is the main dataset used for the remaining analysis steps in this notebook.")

dataset_size = pd.Series(
    {
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
).to_frame("value")

print(f"Filtered dataframe shape: {df.shape}")
display(dataset_size)
display(df.head())
display(pd.Series(df.columns, name="column_name").to_frame())

# Save a tiny CSV sample for quick inspection outside the notebook.
preview_df = df.head(preview_rows).copy()
preview_df.to_csv(preview_csv_path, index=False)
print(f"Exported a {preview_rows}-row CSV preview to: {preview_csv_path}")

missing_summary = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
    .rename("missing_pct")
    .to_frame()
)

display(missing_summary.head(10))

Parquet is the main dataset used for the remaining analysis steps in this notebook.
Filtered dataframe shape: (4886502, 17)


,value
rows,4886502
columns,17


,action_taken,property_value,income,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,state_code,derived_loan_product_type,derived_dwelling_category,loan_purpose,lien_status,occupancy_type,applicant_age,derived_race,derived_sex,derived_ethnicity
0,1,975000,0,5359,91.23,101900,62.0,NJ,Conventional:First Lien,Single Family (1-4 Units):Site-Built,31,1,3,45-54,White,Male,Hispanic or Latino
1,2,655000,140,5395,31.36,156200,117.0,NY,FHA:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,25-34,Asian,Male,Not Hispanic or Latino
2,1,555000,138,7409,97.99,79400,57.0,FL,Conventional:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino
3,2,725000,108,4295,91.13,133300,47.0,NJ,Conventional:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino
4,1,515000,65,2902,41.21,156200,52.0,NY,FHA:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino


,column_name
0,action_taken
1,property_value
2,income
3,tract_population
4,tract_minority_population_percent
5,ffiec_msa_md_median_family_income
6,tract_to_msa_income_percentage
7,state_code
8,derived_loan_product_type
9,derived_dwelling_category


Exported a 20-row CSV preview to: /Users/hsuan/Documents/responsible_ML/Capstone/data/hmda_filtered_preview.csv


,missing_pct
action_taken,0.0
derived_dwelling_category,0.0
derived_sex,0.0
derived_race,0.0
applicant_age,0.0
occupancy_type,0.0
lien_status,0.0
loan_purpose,0.0
derived_loan_product_type,0.0
property_value,0.0


## 7. Label Creation

`action_taken` is converted into a binary target: approve (`1`, `2`) versus deny (`3`). This creates the label used for modeling.

In [17]:
label_map = {"1": 1, "2": 1, "3": 0}

# Map HMDA action outcomes into the binary baseline target.
df["label"] = df["action_taken"].map(label_map)
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

display(df[["action_taken", "label"]].head())
display(df["label"].value_counts().rename_axis("label").to_frame("count"))
display(df["label"].value_counts(normalize=True).rename_axis("label").to_frame("share").round(4))

,action_taken,label
0,1,1
1,2,1
2,1,1
3,2,1
4,1,1


,count
label,
1,3539072
0,1347430


,share
label,
1,0.7243
0,0.2757


## 8. Feature Selection

The model uses a compact set of numeric and categorical features, while `derived_race`, `derived_sex`, and `derived_ethnicity` are kept only for subgroup analysis and not used as training inputs.

In [18]:
fairness_group_columns = ["derived_race", "derived_sex", "derived_ethnicity"]

numeric_features = [
    "property_value",
    "income",
    "tract_population",
    "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income",
    "tract_to_msa_income_percentage",
]

categorical_features = [
    "state_code",
    "derived_loan_product_type",
    "derived_dwelling_category",
    "loan_purpose",
    "lien_status",
    "occupancy_type",
    "applicant_age",
]

# Keep protected attributes for later fairness evaluation, but exclude them from training.
model_features = numeric_features + categorical_features

required_columns = model_features + fairness_group_columns + ["label"]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

feature_summary = pd.DataFrame(
    {
        "feature": numeric_features + categorical_features + fairness_group_columns,
        "role": (
            ["numeric feature"] * len(numeric_features)
            + ["categorical feature"] * len(categorical_features)
            + ["fairness grouping column"] * len(fairness_group_columns)
        ),
    }
)

display(feature_summary)

,feature,role
0,property_value,numeric feature
1,income,numeric feature
2,tract_population,numeric feature
3,tract_minority_population_percent,numeric feature
4,ffiec_msa_md_median_family_income,numeric feature
5,tract_to_msa_income_percentage,numeric feature
6,state_code,categorical feature
7,derived_loan_product_type,categorical feature
8,derived_dwelling_category,categorical feature
9,loan_purpose,categorical feature


## 9. Data Cleaning and Preprocessing

This section keeps the required columns, converts numeric-looking fields to numeric format, standardizes common missing-value tokens, and defines the numeric and categorical preprocessing pipeline.

In [19]:
model_df = df[required_columns].copy()

missing_tokens = {"NA": np.nan, "Exempt": np.nan, "": np.nan}

# Standardize placeholder strings before fitting the preprocessing pipeline.
for column in numeric_features:
    model_df[column] = pd.to_numeric(model_df[column].replace(missing_tokens), errors="coerce")

for column in categorical_features + fairness_group_columns:
    model_df[column] = model_df[column].replace(missing_tokens)

X = model_df[model_features]
y = model_df["label"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

display(model_df[required_columns].head())

,property_value,income,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,state_code,derived_loan_product_type,derived_dwelling_category,loan_purpose,lien_status,occupancy_type,applicant_age,derived_race,derived_sex,derived_ethnicity,label
0,975000.0,0.0,5359,91.23,101900,62.0,NJ,Conventional:First Lien,Single Family (1-4 Units):Site-Built,31,1,3,45-54,White,Male,Hispanic or Latino,1
1,655000.0,140.0,5395,31.36,156200,117.0,NY,FHA:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,25-34,Asian,Male,Not Hispanic or Latino,1
2,555000.0,138.0,7409,97.99,79400,57.0,FL,Conventional:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino,1
3,725000.0,108.0,4295,91.13,133300,47.0,NJ,Conventional:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino,1
4,515000.0,65.0,2902,41.21,156200,52.0,NY,FHA:First Lien,Single Family (1-4 Units):Site-Built,1,1,1,35-44,White,Male,Hispanic or Latino,1


## 10. Train and Test Split

The data is split into training and test sets with `random_state=42` and stratification on the label so the approval-denial balance stays stable across both splits.

In [20]:
# Use a stratified split so the approval rate stays stable across train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

split_summary = pd.Series(
    {
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "train_positive_rate": y_train.mean(),
        "test_positive_rate": y_test.mean(),
    }
).to_frame("value")

display(split_summary.round(4))

,value
train_rows,3.909201e+06
test_rows,9.773010e+05
train_positive_rate,7.243000e-01
test_positive_rate,7.243000e-01


## 11. Baseline Model

This section fits `LogisticRegression(max_iter=1000)` as the baseline model.

In [21]:
# Fit logistic regression as the baseline model.
baseline_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

baseline_pipeline.fit(X_train, y_train)
baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## 12. Evaluation

This section reports a compact baseline summary using `accuracy`, `AUC`, `log-loss`, and a small `FPR` table.

In [22]:
# Baseline evaluation metrics.
y_pred_baseline = baseline_pipeline.predict(X_test)
y_score_baseline = baseline_pipeline.predict_proba(X_test)[:, 1]

baseline_cm = confusion_matrix(y_test, y_pred_baseline, labels=[0, 1])
baseline_tn, baseline_fp, baseline_fn, baseline_tp = baseline_cm.ravel()
baseline_fpr = baseline_fp / (baseline_fp + baseline_tn) if (baseline_fp + baseline_tn) else np.nan

baseline_performance_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred_baseline),
        "auc": roc_auc_score(y_test, y_score_baseline),
        "log_loss": log_loss(y_test, y_score_baseline),
    }
).round(4).to_frame("value")

baseline_fpr_metrics = pd.Series(
    {
        "overall_fpr": baseline_fpr,
    }
).round(4).to_frame("value")

display(baseline_performance_metrics)
display(baseline_fpr_metrics)

,value
accuracy,0.7417
auc,0.7401
log_loss,0.5164


,value
overall_fpr,0.7735


This subgroup evaluation is included as an early diagnostic view of baseline model behavior across groups. At this stage, it is not a formal fairness assessment and should not be treated as a final fairness conclusion.

## 13. Main Model: XGBoost

This section fits `XGBClassifier(random_state=42)` as the main model and reports the same compact evaluation metrics used for the logistic regression baseline.

In [23]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Use XGBoost as the main model pipeline.
model = xgb_pipeline
model.fit(X_train, y_train)

display(model)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

main_cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
main_tn, main_fp, main_fn, main_tp = main_cm.ravel()
main_fpr = main_fp / (main_fp + main_tn) if (main_fp + main_tn) else np.nan

main_performance_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_score),
        "log_loss": log_loss(y_test, y_score),
    }
).round(4).to_frame("value")

main_fpr_metrics = pd.Series(
    {
        "overall_fpr": main_fpr,
    }
).round(4).to_frame("value")

display(main_performance_metrics)
display(main_fpr_metrics)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

,value
accuracy,0.7844
auc,0.8010
log_loss,0.4619


,value
overall_fpr,0.6067


## 14. Second Model: Random Forest

This section fits `RandomForestClassifier(random_state=42)` as a second comparison model. It reuses the existing train/test split and preprocessing output, then reports the same compact evaluation metrics and a three-model comparison table.

In [24]:
# Reuse the fitted preprocessing step from the main model pipeline without building a separate pipeline.
rf_preprocessor = model.named_steps["preprocess"]
X_train_rf = rf_preprocessor.transform(X_train)
X_test_rf = rf_preprocessor.transform(X_test)

# Keep the random forest lighter so it remains practical on the full HMDA training set.
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=25,
    max_features="sqrt",
    max_samples=0.1,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train_rf, y_train)

y_pred_rf = rf_model.predict(X_test_rf)
y_score_rf = rf_model.predict_proba(X_test_rf)[:, 1]

rf_cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
rf_tn, rf_fp, rf_fn, rf_tp = rf_cm.ravel()
rf_fpr = rf_fp / (rf_fp + rf_tn) if (rf_fp + rf_tn) else np.nan

rf_performance_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "auc": roc_auc_score(y_test, y_score_rf),
        "log_loss": log_loss(y_test, y_score_rf),
    }
).round(4).to_frame("value")

rf_fpr_metrics = pd.Series(
    {
        "overall_fpr": rf_fpr,
    }
).round(4).to_frame("value")

display(rf_performance_metrics)
display(rf_fpr_metrics)

comparison_df = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "accuracy": baseline_performance_metrics.loc["accuracy", "value"],
            "auc": baseline_performance_metrics.loc["auc", "value"],
            "log_loss": baseline_performance_metrics.loc["log_loss", "value"],
            "fpr": baseline_fpr_metrics.loc["overall_fpr", "value"],
        },
        {
            "model": "XGBoost",
            "accuracy": main_performance_metrics.loc["accuracy", "value"],
            "auc": main_performance_metrics.loc["auc", "value"],
            "log_loss": main_performance_metrics.loc["log_loss", "value"],
            "fpr": main_fpr_metrics.loc["overall_fpr", "value"],
        },
        {
            "model": "Random Forest",
            "accuracy": rf_performance_metrics.loc["accuracy", "value"],
            "auc": rf_performance_metrics.loc["auc", "value"],
            "log_loss": rf_performance_metrics.loc["log_loss", "value"],
            "fpr": rf_fpr_metrics.loc["overall_fpr", "value"],
        },
    ]
).set_index("model").round(4)

display(comparison_df)

,value
accuracy,0.7653
auc,0.7794
log_loss,0.4879


,value
overall_fpr,0.7626


,accuracy,auc,log_loss,fpr
model,,,,
Logistic Regression,0.7417,0.7401,0.5164,0.7735
XGBoost,0.7844,0.8010,0.4619,0.6067
Random Forest,0.7653,0.7794,0.4879,0.7626
